# Initialize Benchmarking Utilities


In [7]:
# critical imports
import dask.array as da

# simpler imports
import psutil
import random
import threading
import statistics as stat
import time

# dask and coiled imports
import numpy as np
import pandas as pd

import dask.dataframe as dd
import dask.array as da
import dask.bag as db

from dask.distributed import Client, SSHCluster

import coiled
from coiled import Cluster

In [ ]:
# Class to benchmark CPU usage
class benchmarkCPU(threading.Thread):

    # Constructor to initialize the thread
    def __init__(self, intervalS=0.1):
        super().__init__()  # Initialize the thread
        self.running = False  # Control the running state, binary semaphore
        self.name = "unidentified process"
        self.interval = intervalS 
        self.performanceCPU = [] # array of performance (usage) over interval
        self.performanceMem = []
        self.elapsedTime = 0

    # Runs the benchmark when the thread starts
    def run(self):
        self.running = True
        currentProcess = psutil.Process()  # Get the current process
        self.name = currentProcess.name()

        self.elapsedTime = time.time()  # Record the start time of the benchmarking

        while self.running:
            # CPU usage percentage
            utilization = min(currentProcess.cpu_percent(self.interval), 100)
            memoryUt = currentProcess.memory_percent()
            self.performanceCPU.append(utilization) 
            self.performanceMem.append(memoryUt)

    # Stops the benchmark
    def stop(self):
        self.running = False  # Signal loop stop
        self.elapsedTime -= time.time()  # Record the end time of the benchmarking
        self.elapsedTime *= -1
        mCPU = stat.mean(self.performanceCPU)
        mMem = stat.mean(self.performanceMem)*100
        return self.name, round(mCPU,3), round(mMem,3), self.elapsedTime
    
    # generate a benchmarking report
    def report(self):
        print("--Basic Performance Metrics--")
        print("Evaluated with psutil, threading, and time modules.")
        print("-"*20)
        mCPU = stat.mean(self.performanceCPU)
        mMem = stat.mean(self.performanceMem)*100
        print(f"   Process Name:               {self.name}")
        print(f"   avg. CPU utilization:       {round(mCPU,3)}%")
        print(f"   avg. memory utilization:    {round(mMem,3)}%")
        print(f"   elapsed time:               {self.elapsedTime} s")
        print(f"   est. CPU usage time:        {mCPU * self.elapsedTime / 100} s")
        print(f"   est. memory usage time:     {mMem * self.elapsedTime / 100} s")
        print("-"*20)

# ---
def busywork():
    x = da.random.random((50000,50000), chunks=(1000,1000))
    da.exp(x).sum().compute()
    return

# --- start benchmarking
display_cpu = benchmarkCPU(0.1)

display_cpu.start()
try:
    result = busywork()
finally: # stop thread even when I press Ctrl+C
    display_cpu.stop()
    display_cpu.report()

--Basic Performance Metrics--
Evaluated with psutil, threading, and time modules.
--------------------
   Process Name:               python.exe
   avg. CPU utilization:       99.803%
   avg. memory utilization:    106.877%
   elapsed time:               9.505029439926147 s
   est. CPU usage time:        9.486329327658467 s
   est. memory usage time:     10.158670565958088 s
--------------------


In [27]:
""" LOCAL ONLY """
from dask.distributed import Client
client = Client()
client


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 31.75 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:52434,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:52453,Total threads: 2
Dashboard: http://127.0.0.1:52458/status,Memory: 7.94 GiB
Nanny: tcp://127.0.0.1:52437,


In [26]:
client.shutdown()

In [28]:
# --- start benchmarking
display_cpu = benchmarkCPU(0.1)

display_cpu.start()
try:
    result = busywork()
finally: # stop thread even when I press Ctrl+C
    display_cpu.stop()
    display_cpu.report()

--Basic Performance Metrics--
Evaluated with psutil, threading, and time modules.
--------------------
   Process Name:               python.exe
   avg. CPU utilization:       44.079%
   avg. memory utilization:    96.9%
   elapsed time:               16.216896533966064 s
   est. CPU usage time:        7.148198741894383 s
   est. memory usage time:     15.714243915712458 s
--------------------


In [30]:
"""LOCAL CLUSTER - depricated"""
# myHosts = [""]
# worker_options={"nthreads": 6}
# cluster = SSHCluster(hosts=myHosts, worker_options=worker_options)
# client = Client(cluster)
# # Set up the local cluster of machines on my house wifi

'LOCAL CLUSTER - depricated'

In [ ]:
""" DASK WORKER AND SCHEDULER, using coiled """
# start the cluster
cluster = coiled.Cluster(
    n_workers=4,
    region="us-east1",
    worker_memory="8 GiB",
)
client = cluster.get_client()

# submit the work
future = client.submit(busywork)
# start timer after submission
newBench = benchmarkCPU(0.1)
newBench.start()
try:
    answer = future.result() # obtain the result
finally: # stop thread even when I press Ctrl+C
    newBench.stop()
    newBench.report()

print(answer)

[2025-11-20 01:11:55,114][INFO    ][coiled] Fetching latest package priorities...
[2025-11-20 01:11:55,115][INFO    ][coiled.package_sync] Resolving your local Python313 Python environment...
[2025-11-20 01:11:55,233][INFO    ][coiled.package_sync] Scanning 171 python packages...
[2025-11-20 01:11:55,564][INFO    ][coiled] Running pip check...
[2025-11-20 01:11:56,210][INFO    ][coiled] Validating environment...
[2025-11-20 01:12:00,636][INFO    ][coiled] Creating wheel for W:\Academia\McMaster University\Year 4\00 Capstone\001 root\Parallelization Workbench...
[2025-11-20 01:12:00,640][INFO    ][coiled] Creating wheel for ~\AppData\Roaming\Python\Python313\site-packages\win32\lib...
[2025-11-20 01:12:00,645][INFO    ][coiled] Creating wheel for ~\AppData\Roaming\Python\Python313\site-packages\pythonwin...
[2025-11-20 01:12:00,650][INFO    ][coiled] Creating wheel for ~\AppData\Roaming\Python\Python313\site-packages\win32...
[2025-11-20 01:12:00,656][INFO    ][coiled] Uploading coiled_

--Basic Performance Metrics--
Evaluated with psutil, threading, and time modules.
--------------------
   Process Name:               python.exe
   avg. CPU utilization:       9.171%
   avg. memory utilization:    114.572%
   elapsed time:               26.872400283813477 s
   est. CPU usage time:        2.464380949335887 s
   est. memory usage time:     30.78815920078432 s
--------------------
None


In [35]:
cluster.shutdown()

[2025-11-20 01:15:46,118][INFO    ][coiled] Cluster 1274837 deleted successfully.


In [ ]:
# import h5py
# f = h5py.File('result.mat','r')
# data = f.get('data/variable1')
# data = np.array(data) # For converting to a NumPy array
# print(data)

import scipy.io

def formatter(filename):
    mat = scipy.io.loadmat(filename)
    mat = {k:v for k, v in mat.items() if k[0] != '_'}
    data = pd.DataFrame({k: pd.Series(v[0]) for k, v in mat.items()}) # compatible for both python 2.x and python 3.x
    return data

stuff = formatter('result.mat')

print(stuff[0])

                                                None
0  [b'Net', b'MCOS', b'SeriesNetwork', [[37077647...
